# Attention as Kernel Smoothing

Topic 5's recurrent network compressed a whole prefix into $128$ numbers, and its Theorem 5 said how fast that state
forgets. Attention replaces the compression with a weighted lookup over every earlier position. This notebook reads
that lookup as **kernel smoothing**: softmax attention is the Nadaraya–Watson estimator with the kernel
$e^{\langle q, k\rangle/\tau}$, the temperature is a bandwidth, and on the circle the bias–variance trade-off of that
bandwidth is a closed form. Then it puts the lookup to work on text: a kernel cache over a document's own earlier
words, mixed with Topic 3's Kneser–Ney, on held-out synthetic filings and on three real 10-K excerpts.

Every number below is computed by `attention_as_kernel_smoothing.py`, which asserts each claim. The cell below
imports it and shows the splits, regrouped into documents.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "attention_as_kernel_smoothing.py").exists()
                       else here / "notebooks" / "attention-as-kernel-smoothing"))
import attention_as_kernel_smoothing as ak
nlm = ak.nlm

ak.test_documents_partition_corpus()
for split in ak.SPLITS:
    docs = ak.documents(split)
    tokens = [sum(len(s) + 1 for s in d) for d in docs]
    print(f"{split:6} {len(docs):4} documents, {sum(tokens):6,} predicted tokens, median {int(np.median(tokens))} per document")

## 1. A weighted lookup

Rows are tokens. Queries $q_t$, keys $k_j$, and values $v_j$ are linear maps of the inputs, the scores are
$s_{tj} = \langle q_t, k_j\rangle/\tau$ with $\tau = \sqrt{d_k}$, and $o_t = \sum_j \alpha_{tj} v_j$ with
$\alpha_t = \operatorname{softmax}(s_t)$ over the positions $t$ may read.

- **Def 1 (kernel smoother; Nadaraya 1964, Watson 1964).** $\hat m(q) = \sum_j \kappa(q, k_j) v_j / \sum_j \kappa(q, k_j)$.
- **Def 2 (scaled dot-product attention).** $\operatorname{Attn}(Q, K, V) = \operatorname{softmax}(QK^\top/\tau + B)V$,
  with the causal mask $B$.

**Prop 1 (attention is Nadaraya–Watson).** By polarization,
$e^{\langle q,k\rangle/\tau} = e^{\|q\|^2/2\tau}\, e^{\|k\|^2/2\tau}\, e^{-\|q-k\|^2/2\tau}$. The query factor cancels,
so attention is the Gaussian Nadaraya–Watson estimator with bandwidth $\sqrt\tau$ and design weights
$\omega_j = e^{\|k_j\|^2/2\tau}$; with equal key norms it is the unweighted one, and on a sphere it is the von
Mises–Fisher kernel.

**Prop 2 (the mixture posterior).** If $J \sim \pi \propto \omega$, $Q \mid J \sim N(k_J, \tau I)$ and $V = v_J$, then
$P(J = j \mid Q = q) = \alpha_j(q)$ and $E[V \mid Q = q] = o(q)$.

**Prop 3 (order blindness).** Bidirectional attention is permutation-equivariant; one causal layer sees its prefix as a
multiset; two causal layers do not.

In [ ]:
ak.test_attention_is_nadaraya_watson()
ak.test_mixture_posterior()
ak.test_order_blindness()
rng = np.random.default_rng(0)
q, u = rng.normal(size=4), rng.normal(size=4)
R = np.linalg.qr(rng.normal(size=(4, 4)))[0]
K2 = np.stack([q + u, q + R @ u])  # two keys equally far from q, of different lengths
A = ak.attention(q[None], K2, np.eye(2), tau=2.0)[1][0]
print("distances to q:", np.round(np.linalg.norm(K2 - q, axis=1), 6), " key norms²:", np.round((K2 * K2).sum(1), 4))
print(f"weight ratio {A[1] / A[0]:.6f} = e^(Δ‖k‖²/2τ) = {np.exp(((K2[1] @ K2[1]) - (K2[0] @ K2[0])) / 4):.6f}")

A causal layer without positions reads *the dog bit the man* and *the man bit the dog* the same way. The cell below
reverses a held-out prefix, read through Topic 6's word vectors, and compares the last position's output under one
causal layer and under two.

In [ ]:
idx, E = ak.key_vectors()
s = next(s for s in nlm.corpus()["held"] if len(s) >= 8 and all(w in idx for w in s[:8]))
X = E[[idx[w] for w in s[:8]]]
Xr = X[[6, 5, 4, 3, 2, 1, 0, 7]]
rng = np.random.default_rng(1)
one, two = ak.random_layers(rng, 100, 1, 2.0), ak.random_layers(rng, 100, 2, 2.0)
print("prefix:  ", " ".join(s[:8]))
print("reversed:", " ".join(s[i] for i in [6, 5, 4, 3, 2, 1, 0, 7]))
print(f"one layer:  max |Δ output| = {np.abs(ak.causal_stack(X, one)[-1] - ak.causal_stack(Xr, one)[-1]).max():.1e}")
print(f"two layers: max |Δ output| = {np.abs(ak.causal_stack(X, two)[-1] - ak.causal_stack(Xr, two)[-1]).max():.3f}")

## 2. Temperature is bandwidth

**Prop 4 (the two limits).** With $S^*$ the argmax set, $\Delta$ the gap to the runner-up, and $R$ the score range,
$\|o_\tau - \bar v_{S^*}\| \le 2V_{\max}(n - |S^*|)e^{-\Delta/\tau}/|S^*|$ and
$\|o_\tau - \bar v\| \le V_{\max}(e^{R/\tau} - 1)$: a lookup at $\tau \to 0$, the mean at $\tau \to \infty$.

**Thm 1 (attention on the circle is a Fourier multiplier).** With $n$ equispaced keys
$u(\theta_j) = (\cos\theta_j, \sin\theta_j)$ and scores $c\cos(\theta_i - \theta_j)$, the attention matrix is
circulant with eigenvalues
$$\rho_k(c) = \frac{\sum_{m \equiv k\ (n)} I_m(c)}{\sum_{m \equiv 0\ (n)} I_m(c)},$$
and for values $y_j = f(\theta_j) + \varepsilon_j$ the exact risk is
$R(c) = \sum_k (1 - \rho_k)^2 |F_k|^2 + \sigma^2 \sum_{m\equiv 0} I_m(2c)\big/\big(n(\sum_{m\equiv 0} I_m(c))^2\big)$.

**Cor 1.** $c^*(n) \sim (4Bn/\sigma^2\sqrt\pi)^{2/5}$ with $B = \tfrac14\sum_k k^4 \mathrm{power}_k$: angular bandwidth
$n^{-1/5}$, risk $n^{-4/5}$.

In [ ]:
ak.test_temperature_limits()
ak.test_circle_fourier_multiplier()
ak.test_circle_rate()
A = ak.circle_attention(64, 7.3)
print(f"max |FFT eigenvalues − Bessel gains| at n = 64, c = 7.3: {np.abs(np.fft.fft(A[0]).real - ak.circle_gains(64, 7.3)).max():.1e}")
print("gains ρ_1…ρ_6 at c = 7.3:", np.round(ak.circle_gains(64, 7.3, range(1, 7)), 4))
for name, power in ak.SPECTRA.items():
    pre = ak.rate_prefactor(power, ak.SIGMA)
    row = ", ".join(f"n = {n:,}: c* = {ak.optimal_c(n, power, ak.SIGMA):.1f} ({ak.optimal_c(n, power, ak.SIGMA) / (pre * n**0.4):.4f})"
                    for n in (32, 10**4, 10**8))
    print(f"{name:6} c*/(C n^0.4): {row}")

**The trained denoiser.** One attention layer reads tokens $(\cos\theta_j, \sin\theta_j, y_j)$: its queries and keys
see the position only, so its scores are $u_i M u_j^\top$ with $M = W_Q W_K^\top/\sqrt2$, and its values are the
$y_j$. Adam trains it on fresh random functions (a smooth and a rough spectrum, noise $\sigma = \tfrac12$). Rotation
invariance makes the population optimum $M = c^* I$, with $c^*$ from Thm 1.

In [ ]:
ak.test_trained_denoiser()
for cfg in ak.DENOISERS:
    t = ak.train_denoiser(cfg)
    print(f"{cfg.spectrum:6} n = {cfg.n:3}: learned c = {t.c:7.3f}, Thm 1's c* = {t.c_star:7.3f}, "
          f"‖M − c*I‖/‖c*I‖ = {np.linalg.norm(t.M - t.c_star * np.eye(2)) / (t.c_star * 2**0.5):.4f}, "
          f"anisotropy {t.anisotropy:.4f}")

## 3. Why $1/\sqrt d$

**Thm 2.** (i) For independent $q, k$ with i.i.d. coordinates of mean 0 and variance 1, $\operatorname{Var}\langle q, k\rangle = d$.
(ii) The softmax Jacobian $J = \operatorname{diag}\alpha - \alpha\alpha^\top$ (Topic 5's Prop 2) has
$\|J\|_2 \le 1 - \|\alpha\|^2 \le 2(1 - \alpha_{\max})$. (iii) With Gaussian $q$ and keys, unscaled scores saturate the
softmax almost surely as $d \to \infty$, while scaled ones converge to the softmax of i.i.d. standard normals.

In [ ]:
ak.test_scaling_and_saturation()
sc = ak.saturation_curve()
print("d          ", sc["dims"])
for k in ("unscaled_gap", "scaled_gap", "unscaled_jac", "scaled_jac"):
    print(f"{k:12}", [round(v, 4) for v in sc[k]])

## 4. A lookup instead of a state

**Prop 5 (one hop).** For one causal layer and $k < t$,
$\partial o_t/\partial x_k = \alpha_{tk}\,[W_V + \tau^{-1} (W_K q_t) \otimes (v_k - o_t)]$: no product over the positions
in between, only the weight $\alpha_{tk}$, which is $1/t$ when the scores are equal.

**Prop 6 (content addressing).** In Topic 5's long-lag task (read a bit, then $L$ noise symbols, then report it), one
attention layer with uniform weights already outputs $(1 - 2x_0)/(L + 1)$: right at every lag.

In [ ]:
ak.test_one_hop_jacobian()
for L in (10, 60, 600):
    r = ak.lag_construction(L, 50.0)
    print(f"L = {L:4}: output {r['o_bit0']:+.5f} for bit 0, accuracy {r['accuracy']:.0%}, loss {r['loss']:.4f}")

Topic 5 measured the gradient reaching the bit at initialization, $\|\partial\ell/\partial s_0\|$ on batches of
$256$. The same census for one attention layer, at Topic 5's lag of $60$ and across lags, and then training by Topic 5's
protocol (600 Adam steps at lag 60).

In [ ]:
ak.test_lag_attention()
sw = ak.lag_sweep()
print("lag          ", sw["lags"])
for k in ("attention", "elman", "lstm-1", "lstm-3"):
    print(f"{k:12}", " ".join(f"{v:9.2e}" for v in sw[k]))
print("attention × (L + 1):", [f"{a * (L + 1):.2e}" for a, L in zip(sw["attention"], sw["lags"])])
print(ak.lag_attention_training())

## 5. Attention over a document

**Def 3 (kernel cache).** At position $t$ of a document, with preceding token $u_t$ and target $w_t$,
$p_\beta(w \mid t) = \sum_{i<t} \alpha_{ti}\,\mathbb 1[w_i = w]$ with
$\alpha_{ti} \propto \exp(\beta\langle \hat e(u_t), \hat e(u_i)\rangle)$, the keys $\hat e$ being Topic 6's unit SVD
vectors (one coordinate for the sentence start, one per OOV string). The model is
$\lambda\, p_{\mathrm{KN}} + (1 - \lambda)\, p_\beta$, with $(\beta, \lambda)$ chosen on dev.

**Cor 2.** $\beta = 0$ is the unigram document cache (Kuhn and De Mori 1990); $\beta \to \infty$ is the exact cache of
the preceding token's key. Keys are equal for the same word, and for words that Topic 6's vectors cannot tell apart.

In [ ]:
print("words sharing their key with another word:", ak.key_classes())
grid = ak.dev_grid()
cb = ak.cache_benchmark()
print("β       dev λ   dev nats   held nats")
for b in ak.BETAS:
    print(f"{b:6}  {grid[b][0]:.3f}   {grid[b][1]:.6f}   {cb[f'beta={b}']['held']:.6f}")
print("dev chooses (β, λ) =", ak.tune_cache())

The held-out benchmark, with standard errors clustered by document (a cache makes a document's sentences
dependent), by class of target, and by distance back to the previous mention.

In [ ]:
ak.test_kernel_cache()
print(f"KN          {cb['kn'][0]:.6f} nats/token (se {cb['kn'][1]:.4f})")
print(f"KN ⊕ cache  {cb['cache'][0]:.6f} (se {cb['cache'][1]:.4f}); paired gain {cb['gain'][0]:.6f} (se {cb['gain'][1]:.6f})")
print(f"repeat targets ({cb['repeat_share']:.1%}): KN {cb['repeat'][0]:.4f} → {cb['repeat'][1]:.4f};  "
      f"first mentions: {cb['first'][0]:.4f} → {cb['first'][1]:.4f}")
for b in cb["bands"]:
    print(f"  previous mention {b['lo']}–{b['hi'] if b['hi'] < 10**8 else '∞'} back: {b['share']:.1%} of repeats, "
          f"{b['cross_sentence']:.1%} in an earlier sentence, KN {b['kn']:.4f} → {b['cache']:.4f}")
print(f"open vocabulary: KN {cb['kn_bpc'][0]:.6f} bpc → {cb['cache_bpc'][0]:.6f}")

**Def 4 (the learned kernel).** Add a trained bilinear term $\langle F_t A, F_i B\rangle$ to the scores
($A, B \in \mathbb R^{102\times16}$, $F$ the key features) and train it on the training documents by the pointer
likelihood $\sum_t \log\sum_{i<t:\,w_i = w_t}\alpha_{ti}$, which needs no Kneser–Ney. Dev chooses the epoch and $\lambda$.

In [ ]:
ak.test_learned_kernel()
tk, lb = ak.train_learned_kernel(), ak.learned_benchmark()
for i, h in enumerate(tk.history, 1):
    print(f"epoch {i:2}: train pointer {h['train_pointer']:.4f}, dev {h['dev']:.6f} (λ {h['lambda']})" + ("  ← chosen" if i == tk.epoch else ""))
print(f"held out: learned {lb['learned'][0]:.6f} vs fixed {lb['fixed'][0]:.6f}; "
      f"paired {lb['learned_minus_fixed'][0]:.6f} (se {lb['learned_minus_fixed'][1]:.6f})")
print(f"held-out pointer loss: fixed {lb['held_pointer_fixed']:.4f} → learned {lb['held_pointer_learned']:.4f}")

## 6. Kernels that factor

**Prop 7.** One head's log-attention matrix is $QK^\top/\tau - \lambda\mathbb 1^\top$, of rank at most $d_k + 1$; a pattern is
realizable by one head iff a row shift of its logarithm has rank $\le d_k$.

**Prop 8 (positive random features).** $\varphi_w(x) = e^{\langle w, x\rangle - \|x\|^2/2}$ with $w \sim N(0, I)$ gives
$E[\varphi_w(x)\varphi_w(y)] = e^{\langle x,y\rangle}$ with variance $e^{2\langle x,y\rangle}(e^{\|x+y\|^2} - 1)$, so the
relative variance of the kernel $e^{\beta\langle\hat x,\hat y\rangle}$ grows like $e^{4\beta}$.

**Prop 9.** Causal linear attention is a recurrence with a state of fixed size $m(d_v + 1)$: the lookup compressed
back into Topic 5's design.

In [ ]:
ak.test_log_attention_rank()
ak.test_positive_random_features()
ak.test_linear_attention_recurrence()
ak.test_random_feature_cache()
rf = ak.random_feature_benchmark()
print("mean total-variation error of the random-feature cache, held out (m =", ak.RF_MS, ")")
for b, row in rf["tv"].items():
    print(f"  β = {b:3}: " + "  ".join(f"{v:.4f}" for v in row))
print("KN ⊕ random-feature cache at the dev (β, λ):", [round(float(v), 6) for v in rf["nats"]], " exact:", round(float(rf["exact"]), 6))

## 7. Finance case study: the filing as memory

Three public MD&A excerpts (Apple, ExxonMobil, Microsoft), 57.6% of whose tokens the synthetic training text never
saw. Each filing's own earlier words are its cache, with $(\beta, \lambda)$ from synthetic dev. In open-vocabulary bits
per character, an OOV word costs Kneser–Ney its spelling every time; the cache copies it after its first mention.

In [ ]:
ak.test_finance_findings()
f = ak.finance_summary()
for k in ("kn_bpc", "kn_ppm_bpc", "cache_bpc", "learned_bpc", "topic5_best_bpc", "lzma_bpc"):
    print(f"{k:16} {f[k]:.6f}")
print(f"share of the cache's gain on repeated OOV words: {f['gain_share_oov_repeat']:.1%}")
print(f"(diagnostic only: λ tuned on EDGAR itself would be {f['oracle_lambda']}, giving {f['oracle_bpc']:.4f})")
for c in f["curated"]:
    print(f"{c['filing']:10} {c['word']:9} ×{c['count']}: second mention KN {c['kn_bits'][1]:5.1f} bits, cache {c['cache_bits'][1]:5.1f}")

## The lab's numbers, and the full harness

`viz_payload()` holds every number the lab shows. Panel A has the exact risk of every $c$ on the grid, the Fourier gains,
one sample function per case, and the trained $c$ beside $c^*$. Panel B has the reach census and four documents (one
held-out filing and the three EDGAR excerpts) with each variant's bits per token and the curated attention rows.

In [ ]:
ak.test_viz_payload()
P = ak.viz_payload()
for k, case in P["circle"]["cases"].items():
    print(f"{k}: c* = {case['c_star']}, learned {case['c_learned']}, R(c*) = {case['risk_star']}")
for doc in P["cache"]["documents"]:
    print(f"{doc['name']:16} {len(doc['tokens']):4} tokens  bpc " + ", ".join(f"{k} {v}" for k, v in doc["bpc"].items()))

In [ ]:
assert ak._run_harness() == 0